# 02 — Corridor Profiles & Trend Analysis

**Project:** Chicago Road Safety Investment Prioritizer  
**Aligned with:** City of Chicago Vision Zero goals  
**Type:** Read-only corridor trend & anomaly analysis — no source files are modified.  
**Grain:** Corridor-month (43 corridors × 96 months = 4,128 rows, 2018-01 to 2025-12)  

---

This notebook analyzes individual crash trajectories across all 43 high-crash corridors.
It computes 12-month moving averages, classifies corridors into trend categories
(`INCREASING` / `STABLE` / `DECREASING`), and flags statistical anomalies
(corridor-months exceeding mean + 3×std). Every number is computed dynamically.


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
})

ROOT = Path('.').resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print('Project root:', ROOT)


---
## Section 1 — Purpose & Method

**Objective:** Evaluate crash trajectories per corridor over the 96-month panel (2018–2025).

- **12-Month Moving Average:** Smooths seasonal volatility to reveal underlying multi-year direction.
- **Trend Metric (% Change):** Compare average monthly crashes in the last 12 months (2025) vs. first 12 months (2018):
  $$\Delta \% = \frac{\bar{X}_{2025} - \bar{X}_{2018}}{\bar{X}_{2018}} \times 100$$
- **Trend Classification Rule:**
  - `INCREASING`: $\Delta \% > +5.0\%$
  - `DECREASING`: $\Delta \% < -5.0\%$
  - `STABLE`: $-5.0\% \le \Delta \% \le +5.0\%$
- **Anomaly Flag Rule:** Any corridor-month where $X_{t} > \mu_{c} + 3 \sigma_{c}$ (computed per corridor).


In [ ]:
PANEL_PATH    = ROOT / 'data' / 'processed' / 'corridor_month_panel.parquet'
REGISTER_PATH = ROOT / 'data' / 'interim'   / 'high_crash_corridor_register.csv'

df       = pd.read_parquet(PANEL_PATH)
register = pd.read_csv(REGISTER_PATH)

print('Loaded panel:', len(df), 'rows |', df['corridor_id'].nunique(), 'corridors')
print('Date range  :', df['crash_month_start'].min().strftime('%Y-%m'), 'to', df['crash_month_start'].max().strftime('%Y-%m'))


---
## Section 2 — Corridor-Level Summary Table

Summary of total crash burden, KSI burden, 12-month moving average trend, and classification for all 43 corridors.


In [ ]:
summary_rows = []
for (cid, cname), g in df.groupby(['corridor_id', 'corridor_name']):
    g = g.sort_values('crash_month_start').reset_index(drop=True)
    tot = g['total_crashes'].sum()
    ksi = g['ksi_crashes'].sum()
    mean_m = g['total_crashes'].mean()
    std_m  = g['total_crashes'].std()
    
    first12_avg = g.head(12)['total_crashes'].mean()
    last12_avg  = g.tail(12)['total_crashes'].mean()
    pct_change  = (last12_avg - first12_avg) / first12_avg * 100.0 if first12_avg > 0 else 0.0
    
    # OLS slope (crashes per month per month)
    x = np.arange(len(g))
    slope, _ = np.polyfit(x, g['total_crashes'], 1)
    
    if pct_change > 5.0:
        classification = 'INCREASING'
    elif pct_change < -5.0:
        classification = 'DECREASING'
    else:
        classification = 'STABLE'
        
    anomalies = int((g['total_crashes'] > mean_m + 3 * std_m).sum())
    
    summary_rows.append({
        'corridor_id'       : cid,
        'corridor_name'     : cname,
        'total_crashes'     : tot,
        'ksi_crashes'       : ksi,
        'mean_monthly'      : round(mean_m, 2),
        'pct_change_18_25'  : round(pct_change, 1),
        'ols_monthly_slope' : round(slope, 4),
        'classification'    : classification,
        'anomaly_months'    : anomalies,
    })

df_summary = pd.DataFrame(summary_rows).sort_values('total_crashes', ascending=False).reset_index(drop=True)

print('── Trend Classification Counts ────────────────────────────────')
cls_counts = df_summary['classification'].value_counts()
for k, v in cls_counts.items():
    print(f'  {k:<12}: {v:>2} corridors ({v / len(df_summary):.1%})')
print()
print('── Corridor Summary Table (Top 10 by total crashes) ─────────────')
print(df_summary.head(10).to_string(index=False))


In [ ]:
print('── Full 43-Corridor Register & Trend Summary ──────────────────')
print(df_summary.to_string(index=False))


---
## Section 3 — Top 8 Corridors by Trend

The 8 corridors with the highest percentage change in 12-month average crashes (2025 vs. 2018).


In [ ]:
top8_trend = df_summary.sort_values('pct_change_18_25', ascending=False).head(8).reset_index(drop=True)
print('Top 8 corridors by trend (highest % change):')
print(top8_trend[['corridor_id','corridor_name','pct_change_18_25','classification','total_crashes']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for i, row in top8_trend.iterrows():
    cid, cname = row['corridor_id'], row['corridor_name']
    c_df = df[df['corridor_id'] == cid].sort_values('crash_month_start').reset_index(drop=True)
    
    ma12 = c_df['total_crashes'].rolling(12).mean()
    dates = c_df['crash_month_start']
    
    ax = axes[i]
    ax.plot(dates, c_df['total_crashes'], color='#bdc3c7', alpha=0.5, linewidth=1, label='Monthly actual')
    ax.plot(dates, ma12, color='#c0392b' if row['classification'] == 'INCREASING' else '#2980b9',
            linewidth=2.2, label='12-mo moving avg')
    
    ax.set_title(f"{cname} ({cid}) | Trend: {row['pct_change_18_25']:+.1f}% [{row['classification']}]", fontsize=10)
    ax.set_ylabel('Crashes')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Section 3 — Top 8 Corridors by Trend (Highest % Change)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Section 4 — Bottom 8 Corridors by Trend

The 8 corridors with the steepest percentage reduction in crashes (2025 vs. 2018).


In [ ]:
bot8_trend = df_summary.sort_values('pct_change_18_25', ascending=True).head(8).reset_index(drop=True)
print('Bottom 8 corridors by trend (steepest % drop):')
print(bot8_trend[['corridor_id','corridor_name','pct_change_18_25','classification','total_crashes']].to_string(index=False))


In [ ]:
fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)
axes = axes.flatten()

for i, row in bot8_trend.iterrows():
    cid, cname = row['corridor_id'], row['corridor_name']
    c_df = df[df['corridor_id'] == cid].sort_values('crash_month_start').reset_index(drop=True)
    
    ma12 = c_df['total_crashes'].rolling(12).mean()
    dates = c_df['crash_month_start']
    
    ax = axes[i]
    ax.plot(dates, c_df['total_crashes'], color='#bdc3c7', alpha=0.5, linewidth=1, label='Monthly actual')
    ax.plot(dates, ma12, color='#27ae60', linewidth=2.2, label='12-mo moving avg')
    
    ax.set_title(f"{cname} ({cid}) | Trend: {row['pct_change_18_25']:+.1f}% [{row['classification']}]", fontsize=10)
    ax.set_ylabel('Crashes')
    ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Section 4 — Bottom 8 Corridors by Trend (Steepest Drop)', fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


---
## Section 5 — Anomaly Flags

Corridor-months where monthly total crashes exceed $\mu_{c} + 3 \sigma_{c}$ (per-corridor mean plus 3 standard deviations).


In [ ]:
anomaly_records = []
for (cid, cname), g in df.groupby(['corridor_id', 'corridor_name']):
    g = g.sort_values('crash_month_start').reset_index(drop=True)
    mu  = g['total_crashes'].mean()
    sig = g['total_crashes'].std()
    thresh = mu + 3 * sig
    
    anom_rows = g[g['total_crashes'] > thresh]
    for _, r in anom_rows.iterrows():
        anomaly_records.append({
            'corridor_id'       : cid,
            'corridor_name'     : cname,
            'crash_month_start' : r['crash_month_start'].strftime('%Y-%m'),
            'total_crashes'     : r['total_crashes'],
            'corridor_mean'     : round(mu, 1),
            'corridor_std'      : round(sig, 1),
            'threshold_3sigma'  : round(thresh, 1),
            'excess_crashes'    : round(r['total_crashes'] - thresh, 1),
        })

df_anomalies = pd.DataFrame(anomaly_records)
tot_anomaly_months = len(df_anomalies)
top_anom_corridors = df_summary[df_summary['anomaly_months'] > 0][['corridor_id','corridor_name','anomaly_months']].sort_values('anomaly_months', ascending=False)

print(f'Total anomaly corridor-months (total_crashes > mean + 3*std): {tot_anomaly_months}')
print()
print('Corridors with anomaly flags:')
print(top_anom_corridors.to_string(index=False))
print()
print('All 14 flagged anomaly records:')
print(df_anomalies.to_string(index=False))


In [ ]:
# Annotated chart for top anomaly corridor (HCC028 Ashland - 2 anomalies)
ex_cid = 'HCC028'
ex_df = df[df['corridor_id'] == ex_cid].sort_values('crash_month_start').reset_index(drop=True)
ex_name = ex_df['corridor_name'].iloc[0]
mu  = ex_df['total_crashes'].mean()
sig = ex_df['total_crashes'].std()
thresh = mu + 3 * sig

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(ex_df['crash_month_start'], ex_df['total_crashes'], color='#2c3e50', linewidth=1.5, label='Monthly crashes')
ax.axhline(mu, color='#7f8c8d', linestyle='--', linewidth=1, label=f'Mean ({mu:.1f})')
ax.axhline(thresh, color='#e74c3c', linestyle=':', linewidth=1.5, label=f'Mean + 3σ ({thresh:.1f})')

anom_pts = ex_df[ex_df['total_crashes'] > thresh]
ax.scatter(anom_pts['crash_month_start'], anom_pts['total_crashes'], color='#e74c3c', s=70, zorder=5, label='Anomaly flag')
for _, r in anom_pts.iterrows():
    ax.annotate(
        f"{r['crash_month_start'].strftime('%Y-%m')}: {r['total_crashes']} crashes",
        (r['crash_month_start'], r['total_crashes']),
        xytext=(10, 10), textcoords='offset points',
        arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1),
        fontsize=8, fontweight='bold', color='#c0392b'
    )

ax.set_title(f'Annotated Anomaly Chart — {ex_name} ({ex_cid})', fontsize=11, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Total Crashes')
ax.legend(fontsize=8, loc='upper right')
plt.tight_layout()
plt.show()


---
## Section 6 — Limitations

> **This notebook is a read-only analytical exploration of corridor trends.
> It does not represent official City of Chicago policy or engineering approvals.**

1. **Recorded crash burden, not exposure-adjusted risk.** Trajectories reflect total
   recorded crashes per corridor month. They do not control for changes in traffic
   volume (AADT), transit service, or post-pandemic telecommuting patterns.

2. **No volume data.** IDOT/CDOT AADT datasets are not integrated into the monthly panel.
   Downtown corridors (e.g., LaSalle, State, Wacker) show steep crash drops primarily
   reflecting reduced traffic volume, not necessarily improved roadway safety.

3. **Descriptive, not causal.** Observed trends correlate with macro events (COVID-19
   lockdowns, remote work shifts, vehicle design changes) but do not establish causality.

4. **Decision-support only.** Final project prioritization and engineering design remain
   the sole authority of City of Chicago transportation officials.


In [ ]:
print('=' * 60)
print('CORRIDOR PROFILES SUMMARY — Key Verified Findings')
print('=' * 60)
print('  Total corridors analyzed  :', len(df_summary))
print('  Classification counts     :')
for k, v in df_summary['classification'].value_counts().items():
    print(f'    {k:<12}: {v:>2} corridors ({v/len(df_summary):.1%})')
print()
print('  Top 3 steepest INCREASING / Highest trend corridors:')
for _, r in df_summary.sort_values('pct_change_18_25', ascending=False).head(3).iterrows():
    print(f"    {r['corridor_id']} {r['corridor_name']:<20} {r['pct_change_18_25']:>+6.1f}%  [{r['classification']}]")
print()
print('  Top 3 steepest DECREASING corridors:')
for _, r in df_summary.sort_values('pct_change_18_25', ascending=True).head(3).iterrows():
    print(f"    {r['corridor_id']} {r['corridor_name']:<20} {r['pct_change_18_25']:>+6.1f}%  [{r['classification']}]")
print()
print('  Anomaly flags (µ + 3σ)    :', tot_anomaly_months, 'corridor-months across', len(top_anom_corridors), 'corridors')
print('  Top anomaly corridor      :', df_summary.sort_values('anomaly_months', ascending=False).iloc[0]['corridor_id'],
      df_summary.sort_values('anomaly_months', ascending=False).iloc[0]['corridor_name'],
      f"({df_summary.sort_values('anomaly_months', ascending=False).iloc[0]['anomaly_months']} anomaly months)")
print('=' * 60)
